# Part 4 — NumPy for Efficient Numerical Operations
### Data Mondays, Week 4 — Amani Insurance claims case study

**Deliverable:** use **NumPy** for efficient numerical operations.

Big idea: NumPy arrays let you operate on an *entire column of numbers at once*
(**vectorisation**) instead of writing a Python `for` loop over every row. This
isn't just shorter to write — for large datasets it can be 10-100x faster,
because the loop happens inside optimised C code instead of the Python
interpreter.

We'll prove that speed difference on this dataset, then use vectorised NumPy
operations to compute a few real underwriting-style numbers.

In [ ]:
import time

import numpy as np
import pandas as pd

FILE_PATH = "insurance_claims_messy.csv"

In [ ]:
# Quick reused cleaning, same pattern as Part 3.
df = pd.read_csv(FILE_PATH)
df["claim_amount_kes"] = (
    df["claim_amount_kes"].astype(str).str.replace(",", "", regex=False).str.strip()
)
df["claim_amount_kes"] = pd.to_numeric(df["claim_amount_kes"], errors="coerce").abs()
df["claim_type"] = df["claim_type"].astype(str).str.strip().str.lower().str.replace(" ", "-")
group_median = df.groupby("claim_type")["claim_amount_kes"].transform("median")
df["claim_amount_kes"] = df["claim_amount_kes"].fillna(group_median)

# .to_numpy() converts the pandas column into a plain NumPy array of floats --
# this is the object all the vectorised operations below will run on.
amounts = df["claim_amount_kes"].to_numpy()
print(f"Loaded {len(amounts)} claim amounts as a NumPy array of dtype {amounts.dtype}")

## Section 1 — Vectorisation vs. a plain Python loop

Insurers commonly apply a loss-adjustment expense (LAE) loading on top of the
raw claim amount when reserving. We'll apply an 18% loading two ways — once
with a Python `for` loop, once with plain NumPy multiplication — and time both.

In [ ]:
# To make the timing gap obvious, repeat the array 2000 times over,
# simulating a much bigger claims book (~1 million values).
big_amounts = np.tile(amounts, 2000)
print(f"Simulating a bigger book: {len(big_amounts):,} claim amounts")


def loop_version(values):
    result = []
    for v in values:            # one Python-level iteration per value -- slow at scale
        result.append(v * 1.18)
    return result


start = time.perf_counter()
loop_result = loop_version(big_amounts)
loop_time = time.perf_counter() - start

start = time.perf_counter()
vectorised_result = big_amounts * 1.18   # the WHOLE array multiplied in one C-level operation
vector_time = time.perf_counter() - start

print(f"Plain Python loop:     {loop_time:.4f} seconds")
print(f"Vectorised NumPy:      {vector_time:.4f} seconds")
if vector_time > 0:
    print(f"Speedup: ~{loop_time / vector_time:.0f}x faster")

In [ ]:
# Sanity check: both approaches must compute IDENTICAL numbers -- the only
# difference is where the loop happens, not what gets calculated.
print("Spot check, first 3 values match:", np.allclose(loop_result[:3], vectorised_result[:3]))

**What just happened:** both versions compute the exact same numbers. The only
difference is *where* the loop runs — one value at a time in the Python
interpreter (slow), vs the whole array at once inside NumPy's optimised C code
(fast).

## Section 2 — Boolean masking instead of if/else filtering

A **boolean mask** is an array of `True`/`False` values, one per element,
that you can use to filter or select from another array — no loop, no
`if`/`else` needed.

In [ ]:
# A "large claim" reserve-review threshold, e.g. anything over KES 500,000.
# amounts > 500_000 compares EVERY element at once, producing an array of
# True/False the same length as amounts.
large_claim_mask = amounts > 500_000

print(f"large_claim_mask dtype: {large_claim_mask.dtype}, shape: {large_claim_mask.shape}")
print(f"Number of large claims: {large_claim_mask.sum()}  (True counts as 1, so .sum() counts them)")
print(f"Total value of large claims: KES {amounts[large_claim_mask].sum():,.0f}")
print(f"Total value of ALL claims:   KES {amounts.sum():,.0f}")
print(f"Large claims are {amounts[large_claim_mask].sum() / amounts.sum() * 100:.1f}% "
      f"of total claims value, from just {large_claim_mask.mean() * 100:.1f}% of claims.")

**What this shows:** a small share of claims *by count* is driving a much
bigger share of total claims *value* — the classic insurance "80/20" pattern,
now proven with one boolean-mask calculation instead of eyeballing a chart.

## Section 3 — `np.where`: a vectorised if/else

`np.where(condition, value_if_true, value_if_false)` applies an if/else test to
every element of an array at once. Nesting two calls lets us build more than
two outcomes — here, three risk bands.

In [ ]:
# Categorise every claim into a risk band in one line, no loop at all.
risk_band = np.where(
    amounts > 1_000_000, "High",
    np.where(amounts > 200_000, "Medium", "Low")
)

# np.unique with return_counts=True gives us each distinct label plus how
# many times it appears -- a quick way to tally categories from an array.
bands, counts = np.unique(risk_band, return_counts=True)
print("Risk band counts:")
for b, c in zip(bands, counts):
    print(f"  {b:8s} {c}")

## Section 4 — Aggregating across a 2D array with `axis`

So far every array has been 1D (a single column of numbers). NumPy arrays can
also be 2D (rows AND columns) — here, a table of total claims value with one
row per region and one column per month.

In [ ]:
# Parse claim_date into an actual date, then pull out the month number.
df["claim_date_parsed"] = pd.to_datetime(df["claim_date"], errors="coerce", format="mixed")
df["month"] = df["claim_date_parsed"].dt.month
df["region_clean"] = df["region"].astype(str).str.strip().str.title()

# pivot_table builds a region x month table of summed claim values.
# .to_numpy() exposes the genuinely 2D NumPy array sitting underneath it.
pivot = df.pivot_table(
    values="claim_amount_kes", index="region_clean", columns="month",
    aggfunc="sum", fill_value=0,
)
matrix = pivot.to_numpy()
print(f"Matrix shape: {matrix.shape}  (regions x months)")

In [ ]:
# axis=0 collapses ROWS   (moves down each column) -> one total per COLUMN (month)
# axis=1 collapses COLUMNS (moves across each row)   -> one total per ROW (region)
monthly_totals = matrix.sum(axis=0)
region_totals = matrix.sum(axis=1)

print("Total claims value per month, all regions (matrix.sum(axis=0)):")
for month_num, total in zip(pivot.columns, monthly_totals):
    print(f"  Month {month_num:>2}: KES {total:>14,.0f}")

In [ ]:
print("Total claims value per region, all months (matrix.sum(axis=1)):")
for region, total in zip(pivot.index, region_totals):
    print(f"  {region:10s} KES {total:>14,.0f}")

**Mnemonic for `axis`:** `axis=0` moves *down* the rows, so it collapses rows and
leaves one value per **column**. `axis=1` moves *across* the columns, so it
collapses columns and leaves one value per **row**. This trips up almost
everyone the first time — worth re-reading a couple of times.

**Up next:** Part 5 brings in data from *outside* this CSV — a JSON file and a
live currency-exchange API.